# This code is used to import the model and run the model.

pip install ipynb

In [5]:
# Imports
import torch
import torchvision.models as models
import torch.nn as nn
from torchvision import datasets, transforms
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torch.utils.data import Dataset, Subset
import json
import numpy as np
# from ipynb.fs.defs.final_project_V3 import RegNetHead

# Select the model file
* https://stackoverflow.com/questions/9319317/quick-and-easy-file-dialog-in-python/14119223#14119223

In [6]:
# root = tk.Tk()
# root.withdraw()

# model_path = filedialog.askopenfilename(title='Select Model')

# Select the test folder path
* https://stackoverflow.com/questions/50860640/ask-a-user-to-select-folder-to-read-the-files-in-python

In [7]:
# test_file_path = filedialog.askdirectory(title='Select Test Folder')

# Check is there any hardware acceleration

In [8]:
# https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(device)

cuda


# Load the model
* https://pytorch.org/tutorials/beginner/saving_loading_models.html

In [9]:
# Define custom classification head
class RegNetHead(nn.Module):
    def __init__(self, in_features=3712, num_classes=100):  # Adjust `in_features` to match RegNet output
        super(RegNetHead, self).__init__()
         # self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            # nn.Linear(in_features, 1000),
            # nn.ReLU(),
            # nn.Linear(in_features, 1000),
            nn.Linear(in_features, num_classes),
            # nn.Softmax()  # Ensure softmax is applied along the correct dimension
        )

    def forward(self, x):
        # x = self.avgpool(x)
        return self.classifier(x)

# Load pretrained RegNet model
base_model = models.regnet_y_32gf(weights=models.RegNet_Y_32GF_Weights.IMAGENET1K_V2).to(device)

# Remove the original classifier (fc layer) and add our custom head
in_features: int = base_model.fc.in_features  # Get input features of last linear layer
base_model.fc = RegNetHead(in_features, num_classes=100).to(device)  # Replace with custom head

# Verify Model Architecture
print(base_model)

RegNet(
  (stem): SimpleStemIN(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (trunk_output): Sequential(
    (block1): AnyStage(
      (block1-0): ResBottleneckBlock(
        (proj): Conv2dNormActivation(
          (0): Conv2d(32, 232, kernel_size=(1, 1), stride=(2, 2), bias=False)
          (1): BatchNorm2d(232, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (f): BottleneckTransform(
          (a): Conv2dNormActivation(
            (0): Conv2d(32, 232, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): BatchNorm2d(232, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): ReLU(inplace=True)
          )
          (b): Conv2dNormActivation(
            (0): Conv2d(232, 232, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
            (1):

# Load weights

In [10]:
FILE_PATH: str = "./weights/20250310_053536.pth"

base_model.load_state_dict(torch.load(FILE_PATH, weights_only=True,map_location=torch.device(device)))

<All keys matched successfully>

# Transfrom image

In [11]:
transform = transforms.Compose([
    transforms.Resize((224,244)),
    transforms.ToTensor(),
])

# Load test 

In [12]:
TEST_DIR_PATH: str = "./train/train"

dataset: datasets.ImageFolder = datasets.ImageFolder(TEST_DIR_PATH, transform=transform)
loader: DataLoader = DataLoader(dataset, shuffle=False)

In [ ]:
RESULT_CSV_PATH: str = "pumping_lemma_proof.csv"

with open(RESULT_CSV_PATH, 'w') as f:
    f.write("ID,Label\n")

i: int = 0
for data in loader:
    image, label = data
    image, label = image.to(device=device), label.to(device=device)
    with torch.no_grad():
        output = base_model(image)

    _, predicted = torch.max(output.data, 1)

    predicted: int = predicted.to('cpu').numpy()[0]
    with open(RESULT_CSV_PATH, 'a') as f:
        f.write(f"{i}.jpg,{predicted}\n")

    i += 1